# POMDP Benchmark Comparison

Compare quantum QBRL planner against classical baselines:
- POMCP (Monte Carlo Planning)
- PBVI (Point-Based Value Iteration)
- DESPOT (Determinized Sparse Partially Observable Tree)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from quantum_common.visualization.styles import apply_publication_style, QUANTUM_COLORS

apply_publication_style()

from quantum_pomdp.scenarios.tiger_problem import create_tiger_pomdp
from quantum_pomdp.models.belief_state import BeliefState
from quantum_pomdp.algorithms.qbrl import QBRLPlanner, QBRLConfig
from quantum_pomdp.classical_baselines.pomcp import POMCPSolver
from quantum_pomdp.analysis.metrics import PlanningMetrics

In [ ]:
model = create_tiger_pomdp(listen_accuracy=0.85)

# QBRL (classical mode for benchmarking)
qbrl_config = QBRLConfig(horizon=2, num_samples=200, use_quantum=False)
qbrl = QBRLPlanner(model, qbrl_config)

# POMCP baseline
pomcp = POMCPSolver(num_simulations=500, max_depth=10)

belief = BeliefState.uniform(2)
qbrl_action = qbrl.select_action(belief)
pomcp_action = pomcp.select_action(belief.probabilities, model)

print(f"QBRL action: {qbrl_action}")
print(f"POMCP action: {pomcp_action}")

In [ ]:
# Benchmark: compute action quality over multiple episodes
import time

n_episodes = 50
results = {'QBRL': [], 'POMCP': []}
times = {'QBRL': [], 'POMCP': []}
rng = np.random.default_rng(42)

for ep in range(n_episodes):
    b = BeliefState.uniform(2)
    
    t0 = time.perf_counter()
    a_qbrl = qbrl.select_action(b)
    times['QBRL'].append(time.perf_counter() - t0)
    results['QBRL'].append(model.expected_reward(b.probabilities, a_qbrl))
    
    t0 = time.perf_counter()
    a_pomcp = pomcp.select_action(b.probabilities, model)
    times['POMCP'].append(time.perf_counter() - t0)
    results['POMCP'].append(model.expected_reward(b.probabilities, a_pomcp))

print(f"QBRL  - mean reward: {np.mean(results['QBRL']):.3f}, mean time: {np.mean(times['QBRL'])*1000:.1f}ms")
print(f"POMCP - mean reward: {np.mean(results['POMCP']):.3f}, mean time: {np.mean(times['POMCP'])*1000:.1f}ms")

In [ ]:
# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(results['QBRL'], alpha=0.7, label='QBRL', color=QUANTUM_COLORS['quantum'])
ax1.hist(results['POMCP'], alpha=0.7, label='POMCP', color=QUANTUM_COLORS['classical'])
ax1.set_xlabel('Expected Reward')
ax1.set_ylabel('Count')
ax1.set_title('Action Quality Distribution')
ax1.legend()

methods = ['QBRL', 'POMCP']
mean_times = [np.mean(times[m])*1000 for m in methods]
colors = [QUANTUM_COLORS['quantum'], QUANTUM_COLORS['classical']]
ax2.bar(methods, mean_times, color=colors)
ax2.set_ylabel('Time (ms)')
ax2.set_title('Mean Decision Time')

plt.tight_layout()
plt.show()